# USB and HID — enumerate, control, bulk, reports

Raw USB transfers and HID reports through one API. The cells use a virtual bus; replace the backend with `NativeUsbBackend.Instance` (the default) to reach real devices.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.15.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.15.0-preview.1"

## Enumerate

In [ ]:
using System.Text;
using IoTCom.Net.Transport.Usb;

var usbBus = new VirtualUsbBus();
usbBus.Add(new VirtualLoopbackDevice());
var boardSim = usbBus.AddHid(new VirtualHidRelayBoard(relays: 4));
foreach (var d in UsbDevice.List(usbBus)) Console.WriteLine($"{d.Id}  {d.Manufacturer} {d.Product}  [{string.Join(", ", d.Interfaces.Select(i => i.ClassName))}]");
foreach (var h in HidDevice.List(usbBus)) Console.WriteLine($"{h.VendorId:x4}:{h.ProductId:x4}  {h.Product}  {h.UsageName}");
Console.WriteLine(UsbIds.Known(0x1D50, 0x606F));

## Control and bulk transfers
A vendor request reads the firmware version; data written to bulk OUT 0x01 comes back on bulk IN 0x81.

In [ ]:
var loopDev = UsbDevice.Create(o => { o.UseVirtual(usbBus); o.DeviceId = "1209:0001"; });
await loopDev.ConnectAsync();
Console.WriteLine(Encoding.ASCII.GetString(await loopDev.ControlInAsync(UsbSetup.Vendor(0x01), 16)));
await loopDev.WriteAsync(0x01, Encoding.ASCII.GetBytes("ping"));
Console.WriteLine(Encoding.ASCII.GetString((await loopDev.ReadAsync(0x81))!));
await loopDev.DisposeAsync();

## A USB HID relay board
Feature report `00 FF n` switches relay n on; reading feature report 0 returns the serial and the relay bits.

In [ ]:
var relayHid = HidDevice.Create(o => { o.UseVirtual(usbBus); o.UseDevice(HidRelayBoard.VendorId, HidRelayBoard.ProductId); });
await relayHid.ConnectAsync();
var relayBoard = new HidRelayBoard(relayHid);
await relayBoard.SetAsync(3, true);
Console.WriteLine($"{await relayBoard.GetSerialAsync()}: {string.Join(" ", await relayBoard.GetStatesAsync())} (bits {boardSim.State})");
var readOnlyHid = HidDevice.Create(o => { o.UseVirtual(usbBus); o.UseDevice(HidRelayBoard.VendorId, HidRelayBoard.ProductId); o.ReadOnly = true; });
await readOnlyHid.ConnectAsync();
try { await new HidRelayBoard(readOnlyHid).SetAllAsync(true); } catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }

## Going further
`iotcom usb list`, `iotcom usb hid`, `iotcom usb relay --sim`, the Gallery's *USB bench* and the UsbRelay sample. See `docs/en/protocols/usb.md` for drivers and permissions.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*